# 01 — Intent dataset exploration

This notebook helps you **understand a dataset before training the model**. It covers splits, columns, labels, class distributions, and real examples.

Notebooks are useful for exploration and interactive experiments. Production training should run from `scripts/` so it remains easy to reproduce.

## 1. Select the dataset

Change `DATASET_KEY` to `"clinc_oos"` to explore the second dataset. The dictionary centralizes the Hugging Face identifier, dataset configuration, and column names.

In [ ]:
from collections import Counter

from datasets import load_dataset

DATASETS = {
    "banking77": {
        "path": "PolyAI/banking77",
        "name": None,
        "text_column": "text",
        "label_column": "label",
    },
    "clinc_oos": {
        "path": "clinc/clinc_oos",
        "name": "plus",
        "text_column": "text",
        "label_column": "intent",
    },
}

DATASET_KEY = "banking77"
dataset_config = DATASETS[DATASET_KEY]
dataset = load_dataset(dataset_config["path"], dataset_config["name"])
dataset

## 2. Review splits and example counts

This confirms which training, validation, and test sets are available and reveals important size differences.

In [ ]:
for split_name, split in dataset.items():
    print(f"{split_name:>12}: {len(split):,} examples")

## 3. Inspect columns and examples

Before writing preprocessing code, we need to know the column names and types. Reading real examples also clarifies the input the model will receive.

In [ ]:
train = dataset["train"]

print("Columns:", train.column_names)
print("Types:", train.features)
print("First example:", train[0])

## 4. Identify text, labels, and class names

Banking77 uses `text` and `label`; CLINC OOS uses `text` and `intent`. The selected configuration handles this difference without changing the rest of the notebook.

In [ ]:
TEXT_COLUMN = dataset_config["text_column"]
LABEL_COLUMN = dataset_config["label_column"]

label_feature = train.features[LABEL_COLUMN]
label_names = getattr(label_feature, "names", None)

print("Number of classes:", len(label_names) if label_names else "not specified")
print("First classes:", label_names[:10] if label_names else "no names available")

## 5. Analyze class balance

A highly imbalanced distribution may require metrics beyond accuracy, special sampling, or class weights.

In [ ]:
counts = Counter(train[LABEL_COLUMN])
rows = []

for label_id, count in counts.most_common():
    label_name = label_names[label_id] if label_names else str(label_id)
    rows.append((label_id, label_name, count))

print("Classes with the most examples:")
for row in rows[:10]:
    print(row)

print("\nClasses with the fewest examples:")
for row in rows[-10:]:
    print(row)

## 6. Read examples with human-readable labels

This manual review helps identify similar classes, ambiguous text, noise, and possible labeling errors.

In [ ]:
for example in train.select(range(min(10, len(train)))):
    label_id = example[LABEL_COLUMN]
    label_name = label_names[label_id] if label_names else str(label_id)
    print(f"[{label_name}] {example[TEXT_COLUMN]}")

## 7. Run basic quality checks

These checks look for empty and duplicate texts. They are simple safeguards against easily detectable training-data problems.

In [ ]:
texts = train[TEXT_COLUMN]
empty_count = sum(not text or not text.strip() for text in texts)
duplicate_count = len(texts) - len(set(texts))

print("Empty texts:", empty_count)
print("Duplicate texts:", duplicate_count)

## Next step

After understanding the data, move final decisions into `configs/datasets/` and reproducible preprocessing into `src/qwen3_finetuning/data/`. The notebook should not become the only place where project logic lives.